In [1]:
import numpy as np
import pandas as pd

import lightgbm as lgb
from xgboost import XGBRegressor

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings("ignore")

In [2]:
PATH = "/kaggle/input/competitions/datathon-2026-round-1/"

sales = pd.read_csv(PATH+"sales.csv", parse_dates=["Date"])
test  = pd.read_csv(PATH+"sample_submission.csv", parse_dates=["Date"])
promo = pd.read_csv(PATH+"promotions.csv", parse_dates=["start_date","end_date"])
web   = pd.read_csv(PATH+"web_traffic.csv", parse_dates=["date"])
inv   = pd.read_csv(PATH+"inventory.csv", parse_dates=["snapshot_date"])

In [3]:
sales["is_test"] = 0
test["is_test"] = 1

test["Revenue"] = np.nan
test["COGS"] = np.nan

df = pd.concat([sales, test], ignore_index=True)

In [4]:
df["dow"]   = df.Date.dt.dayofweek
df["month"] = df.Date.dt.month
df["dom"]   = df.Date.dt.day
df["doy"]   = df.Date.dt.dayofyear
df["year"]  = df.Date.dt.year

df["weekend"] = (df["dow"] >= 5).astype(int)

df["trend"] = (df["Date"] - df["Date"].min()).dt.days

# CAP trend → avoid 2024 explosion
max_trend = df[df["is_test"]==0]["trend"].max()
df["trend"] = df["trend"].clip(upper=max_trend)

# Fourier (low order only)
df["sin_year"] = np.sin(2*np.pi*df["doy"]/365)
df["cos_year"] = np.cos(2*np.pi*df["doy"]/365)

In [5]:
tet_map = {
    2022:"2022-02-01",
    2023:"2023-01-22",
    2024:"2024-02-10"
}

tet = pd.DataFrame(list(tet_map.items()), columns=["year","tet"])
tet["tet"] = pd.to_datetime(tet["tet"])

df = df.merge(tet, on="year", how="left")

df["days_to_tet"] = (df["tet"] - df["Date"]).dt.days

df["tet_weight"] = np.exp(-0.1 * df["days_to_tet"].clip(0,30))
df["tet_weight"] = df["tet_weight"].clip(0,1)

In [6]:
calendar = pd.DataFrame({"Date": pd.date_range(df.Date.min(), df.Date.max())})
calendar["promo"] = 0
calendar["promo_strength"] = 0

for _, r in promo.iterrows():
    mask = (calendar.Date >= r.start_date) & (calendar.Date <= r.end_date)

    calendar.loc[mask, "promo"] = 1

    if r["promo_type"] == "percentage":
        calendar.loc[mask, "promo_strength"] = np.maximum(
            calendar.loc[mask, "promo_strength"], r["discount_value"]
        )

calendar["promo_strength"] = np.log1p(calendar["promo_strength"])

df = df.merge(calendar, on="Date", how="left")

In [7]:
hist = df[df["is_test"]==0]

base = hist.groupby(["month","dow"])["Revenue"].mean().reset_index()
base.rename(columns={"Revenue":"base_rev"}, inplace=True)

df = df.merge(base, on=["month","dow"], how="left")

global_mean = hist["Revenue"].mean()
df["base_rev"] = df["base_rev"].fillna(global_mean)

df["base_log"] = np.log1p(df["base_rev"])

In [8]:
web["dow"] = web["date"].dt.dayofweek
web["month"] = web["date"].dt.month

traffic = web.groupby(["month","dow"])["sessions"].mean().reset_index()
traffic.rename(columns={"sessions":"traffic"}, inplace=True)

df = df.merge(traffic, on=["month","dow"], how="left")

df["traffic"] = df["traffic"].fillna(df["traffic"].mean())
df["traffic_log"] = np.log1p(df["traffic"])

In [9]:
train = df[df["is_test"]==0].copy()
test  = df[df["is_test"]==1].copy()

train["target"] = np.log1p(train["Revenue"])

In [10]:
features = [
    "dow","month","dom",
    "trend","weekend",
    "sin_year","cos_year",
    "promo","promo_strength",
    "tet_weight",
    "base_log",
    "traffic_log"
]

In [11]:
features = [
    "dow","month","dom",
    "trend","weekend",
    "sin_year","cos_year",
    "promo","promo_strength",
    "tet_weight",
    "base_log",
    "traffic_log"
]

In [12]:
lgb_params = dict(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.5,
    random_state=42
)

xgb_params = dict(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.5,
    tree_method="hist",
    random_state=42
)

In [13]:
tscv = TimeSeriesSplit(n_splits=5)

lgb_models = []
xgb_models = []

for tr, va in tscv.split(train):

    X_tr = train.iloc[tr][features]
    y_tr = train.iloc[tr]["target"]

    X_va = train.iloc[va][features]
    y_va = train.iloc[va]["target"]

    lgb_model = lgb.LGBMRegressor(**lgb_params)
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_va,y_va)],
        callbacks=[lgb.early_stopping(80, verbose=False)]
    )

    xgb_model = XGBRegressor(**xgb_params)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va,y_va)], verbose=False)

    lgb_models.append(lgb_model)
    xgb_models.append(xgb_model)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001680 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 846
[LightGBM] [Info] Number of data points in the train set: 643, number of used features: 11
[LightGBM] [Info] Start training from score 15.198360
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

In [14]:
def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

val_cut = "2022-01-01"

val = train[train["Date"] >= val_cut]

pred_val = np.mean([
    np.expm1(m.predict(val[features]))
    for m in lgb_models
], axis=0)

print("Validation WAPE:", wape(val["Revenue"], pred_val))

Validation WAPE: 0.342556076305504


In [15]:
lgb_preds = np.mean([
    np.expm1(m.predict(test[features]))
    for m in lgb_models
], axis=0)

xgb_preds = np.mean([
    np.expm1(m.predict(test[features]))
    for m in xgb_models
], axis=0)

# 🔥 tuned blend
preds = 0.6*lgb_preds + 0.4*xgb_preds

preds = np.clip(preds, 0, None)

test["Revenue"] = preds
test["COGS"] = np.minimum(preds*0.75, preds)

In [16]:
sub = test[["Date","Revenue","COGS"]].copy()
sub["Date"] = sub["Date"].dt.strftime("%Y-%m-%d")

sub.to_csv("submission_SOTA.csv", index=False)
print("DONE")

DONE
